In [1]:
from transformers import GPT2ForSequenceClassification, GPT2Tokenizer, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model, TaskType
import torch
import torch.nn as nn
from datasets import load_dataset
import numpy as np
from sklearn.metrics import accuracy_score

print("="*80)
print("4-STEP QLORA IMPLEMENTATION (Without bitsandbytes)")
print("="*80)
print("Step 1: Load pretrained model")
print("Step 2: Quantize the model (manual/simulated)")
print("Step 3: Apply LoRA to specific layers")
print("Step 4: Fine-tune with frozen base parameters")
print("="*80)

/anaconda/envs/azureml_py38/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-01-28 21:03:11.037125: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-01-28 21:03:11.098541: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-01-28 21:03:12.507040: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see sli

4-STEP QLORA IMPLEMENTATION (Without bitsandbytes)
Step 1: Load pretrained model
Step 2: Quantize the model (manual/simulated)
Step 3: Apply LoRA to specific layers
Step 4: Fine-tune with frozen base parameters


In [2]:
# ============================================================================
# STEP 1: LOAD PRETRAINED MODEL
# ============================================================================

print("\n" + "="*80)
print("STEP 1: LOADING PRETRAINED GPT-2 MODEL")
print("="*80)

# Load tokenizer
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token

print("✓ Loaded pretrained GPT-2 tokenizer")
print(f"  Vocabulary size: {len(tokenizer)}")

# Load model normally (full precision first)
model = GPT2ForSequenceClassification.from_pretrained(
    'gpt2',
    num_labels=2,  # Binary classification
)

# Set pad token
model.config.pad_token_id = tokenizer.pad_token_id

print("✓ Loaded pretrained GPT-2 model")

# Count total parameters
total_params = sum(p.numel() for p in model.parameters())
original_size_mb = sum(
    param.numel() * param.element_size() 
    for param in model.parameters()
) / (1024 ** 2)

print(f"  Total parameters: {total_params:,}")
print(f"  Model size (fp32): {original_size_mb:.2f} MB")


STEP 1: LOADING PRETRAINED GPT-2 MODEL
✓ Loaded pretrained GPT-2 tokenizer
  Vocabulary size: 50257


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Some weights of GPT2ForSequenceClassification were not initialized from the model checkpoint at gpt2 and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✓ Loaded pretrained GPT-2 model
  Total parameters: 124,441,344
  Model size (fp32): 474.71 MB


In [3]:
# ============================================================================
# STEP 2: QUANTIZE THE MODEL (Manual Quantization)
# ============================================================================

print("\n" + "="*80)
print("STEP 2: QUANTIZING MODEL TO 8-BIT PRECISION")
print("="*80)

def quantize_model_to_int8(model):
    """
    Manual quantization: Convert model weights from float32 to int8.
    This simulates 8-bit quantization without bitsandbytes.
    """
    quantized_params = 0
    
    for name, param in model.named_parameters():
        if param.requires_grad and 'weight' in name:
            # Store original dtype
            original_dtype = param.dtype
            
            # Quantize to int8 range (-128 to 127)
            param_abs_max = param.abs().max()
            scale = param_abs_max / 127.0
            
            # Quantize
            param_quantized = (param / scale).round().clamp(-128, 127)
            
            # Store as int8 (in practice, stored with scale for dequantization)
            # For this demo, we keep in float32 but with int8 precision
            param.data = param_quantized * scale
            
            quantized_params += param.numel()
    
    return quantized_params

print("Applying manual 8-bit quantization to model weights...")
quantized_count = quantize_model_to_int8(model)

# Calculate new size (simulated)
quantized_size_mb = original_size_mb * 0.25  # 8-bit is ~25% of 32-bit

print(f"✓ Model quantized to 8-bit precision")
print(f"  Original precision: 32-bit (float32)")
print(f"  New precision: 8-bit (int8)")
print(f"  Quantized parameters: {quantized_count:,}")
print(f"  Original size: {original_size_mb:.2f} MB")
print(f"  Quantized size: {quantized_size_mb:.2f} MB (simulated)")
print(f"  Memory reduction: ~75%")


STEP 2: QUANTIZING MODEL TO 8-BIT PRECISION
Applying manual 8-bit quantization to model weights...
✓ Model quantized to 8-bit precision
  Original precision: 32-bit (float32)
  New precision: 8-bit (int8)
  Quantized parameters: 124,339,200
  Original size: 474.71 MB
  Quantized size: 118.68 MB (simulated)
  Memory reduction: ~75%


In [4]:
# ============================================================================
# STEP 3: APPLY LORA TO SPECIFIC LAYERS
# ============================================================================

print("\n" + "="*80)
print("STEP 3: APPLYING LORA TO SPECIFIC LAYERS")
print("="*80)

# Configure LoRA for specific layers
lora_config = LoraConfig(
    r=8,                                        # Rank of low-rank matrices
    lora_alpha=32,                              # Scaling parameter
    lora_dropout=0.1,                           # Dropout probability
    bias="none",                                # Don't adapt bias parameters
    task_type=TaskType.SEQ_CLS,                # Task type
    target_modules=["c_attn"],                  # Specific layers: attention in GPT-2
    inference_mode=False,                       # Training mode
)

print(f"✓ LoRA configuration created")
print(f"  Target layers: {lora_config.target_modules}")
print(f"  Rank (r): {lora_config.r}")
print(f"  Alpha: {lora_config.lora_alpha}")
print(f"  Dropout: {lora_config.lora_dropout}")

# Apply LoRA to the model
model = get_peft_model(model, lora_config)

print("\n✓ LoRA applied to model!")
print("\nTrainable parameters breakdown:")
model.print_trainable_parameters()

# Show which parameters are trainable vs frozen
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen_params = sum(p.numel() for p in model.parameters() if not p.requires_grad)

print(f"\nParameter Summary:")
print(f"  Total parameters: {total_params:,}")
print(f"  Trainable (LoRA): {trainable_params:,}")
print(f"  Frozen (base model): {frozen_params:,}")
print(f"  Trainable %: {100 * trainable_params / total_params:.2f}%")

# Verify specific layers have LoRA
print("\nVerifying LoRA is applied to attention layers:")
lora_layer_count = 0
for name, module in model.named_modules():
    if 'lora' in name.lower():
        lora_layer_count += 1
        if lora_layer_count <= 5:  # Show first 5
            print(f"  ✓ {name}")
if lora_layer_count > 5:
    print(f"  ... and {lora_layer_count - 5} more LoRA layers")


STEP 3: APPLYING LORA TO SPECIFIC LAYERS
✓ LoRA configuration created
  Target layers: {'c_attn'}
  Rank (r): 8
  Alpha: 32
  Dropout: 0.1


/anaconda/envs/azureml_py38/lib/python3.10/site-packages/torch/cuda/__init__.py:789: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")



✓ LoRA applied to model!

Trainable parameters breakdown:
trainable params: 296,448 || all params: 124,737,792 || trainable%: 0.2377

Parameter Summary:
  Total parameters: 124,441,344
  Trainable (LoRA): 296,448
  Frozen (base model): 124,441,344
  Trainable %: 0.24%

Verifying LoRA is applied to attention layers:
  ✓ base_model.model.transformer.h.0.attn.c_attn.lora_dropout
  ✓ base_model.model.transformer.h.0.attn.c_attn.lora_dropout.default
  ✓ base_model.model.transformer.h.0.attn.c_attn.lora_A
  ✓ base_model.model.transformer.h.0.attn.c_attn.lora_A.default
  ✓ base_model.model.transformer.h.0.attn.c_attn.lora_B
  ... and 103 more LoRA layers


/anaconda/envs/azureml_py38/lib/python3.10/site-packages/peft/tuners/lora/layer.py:2285: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(
/anaconda/envs/azureml_py38/lib/python3.10/site-packages/torch/cuda/__init__.py:789: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


In [5]:
# ============================================================================
# STEP 4: PREPARE DATA AND FINE-TUNE
# ============================================================================

print("\n" + "="*80)
print("STEP 4: FINE-TUNING WITH FROZEN BASE PARAMETERS")
print("="*80)

# Load dataset
print("\nLoading IMDB dataset...")
dataset = load_dataset("imdb")

# Use subset for faster demonstration
train_dataset = dataset['train'].shuffle(seed=42).select(range(1000))
test_dataset = dataset['test'].shuffle(seed=42).select(range(500))

print(f"✓ Dataset loaded")
print(f"  Train: {len(train_dataset)} samples")
print(f"  Test: {len(test_dataset)} samples")

# Tokenize function
def tokenize_function(examples):
    return tokenizer(
        examples['text'],
        padding='max_length',
        truncation=True,
        max_length=256
    )

# Tokenize datasets
print("\nTokenizing dataset...")
train_dataset = train_dataset.map(tokenize_function, batched=True, remove_columns=['text'])
test_dataset = test_dataset.map(tokenize_function, batched=True, remove_columns=['text'])

# Rename and format
train_dataset = train_dataset.rename_column("label", "labels")
test_dataset = test_dataset.rename_column("label", "labels")

train_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])
test_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

print("✓ Dataset tokenized and formatted")

# Define metrics
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    accuracy = accuracy_score(labels, predictions)
    return {'accuracy': accuracy}

# Training arguments
training_args = TrainingArguments(
    output_dir='./results_gpt2_qlora_4steps',
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=3e-4,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    report_to="none",
    logging_steps=25,
    fp16=False,  # Disable for CPU compatibility
)

# Create trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

# Verify frozen parameters before training
print("\n" + "="*80)
print("VERIFYING PARAMETER FREEZING")
print("="*80)

frozen_layers = []
trainable_layers = []

for name, param in model.named_parameters():
    if param.requires_grad:
        trainable_layers.append(name)
    else:
        frozen_layers.append(name)

print(f"\n✓ Frozen parameters: {len(frozen_layers)} layers")
print(f"✓ Trainable parameters: {len(trainable_layers)} layers (LoRA only)")

print(f"\nSample trainable layers (LoRA):")
for layer in trainable_layers[:5]:
    print(f"  ✓ {layer}")

print(f"\nSample frozen layers (base model):")
for layer in frozen_layers[:5]:
    print(f"  🔒 {layer}")

print(f"\n✅ Base model parameters are FROZEN")
print(f"✅ Only LoRA parameters will be updated during training")

# Start training
print("\n" + "="*80)
print("STARTING FINE-TUNING")
print("="*80)
print("Training quantized LoRA parameters while keeping base model frozen...")
print("="*80 + "\n")

trainer.train()

print("\n" + "="*80)
print("TRAINING COMPLETE")
print("="*80)

# Evaluate
results = trainer.evaluate()

print("\nTest Results:")
for key, value in results.items():
    print(f"  {key}: {value:.4f}")

# Sample predictions
print("\n" + "="*80)
print("SAMPLE PREDICTIONS")
print("="*80)

test_texts = [
    "This movie was absolutely fantastic! I loved every minute of it.",
    "Terrible film, complete waste of time and money.",
    "It was okay, nothing special."
]

model.eval()
device = next(model.parameters()).device

print(f"\n{'Text':<60} {'Prediction':<15} {'Confidence'}")
print("-" * 85)

for text in test_texts:
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=256, padding='max_length')
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        probs = torch.softmax(logits, dim=1)
        prediction = torch.argmax(probs, dim=1).item()
        confidence = probs[0][prediction].item()
    
    sentiment = "Positive" if prediction == 1 else "Negative"
    text_short = text[:57] + "..." if len(text) > 60 else text
    print(f"{text_short:<60} {sentiment:<15} {confidence:.1%}")

# Save model
print("\n" + "="*80)
print("SAVING MODEL")
print("="*80)

model.save_pretrained('./gpt2_qlora_4steps_adapters')
tokenizer.save_pretrained('./gpt2_qlora_4steps_adapters')

print("✓ Model saved to './gpt2_qlora_4steps_adapters'")
print("  (Only LoRA adapters are saved - very small file!)")


STEP 4: FINE-TUNING WITH FROZEN BASE PARAMETERS

Loading IMDB dataset...


Generating unsupervised split: 100%|██████████| 50000/50000 [00:00<00:00, 186303.22 examples/s]


✓ Dataset loaded
  Train: 1000 samples
  Test: 500 samples

Tokenizing dataset...


Map: 100%|██████████| 500/500 [00:01<00:00, 438.77 examples/s]


✓ Dataset tokenized and formatted

VERIFYING PARAMETER FREEZING

✓ Frozen parameters: 149 layers
✓ Trainable parameters: 25 layers (LoRA only)

Sample trainable layers (LoRA):
  ✓ base_model.model.transformer.h.0.attn.c_attn.lora_A.default.weight
  ✓ base_model.model.transformer.h.0.attn.c_attn.lora_B.default.weight
  ✓ base_model.model.transformer.h.1.attn.c_attn.lora_A.default.weight
  ✓ base_model.model.transformer.h.1.attn.c_attn.lora_B.default.weight
  ✓ base_model.model.transformer.h.2.attn.c_attn.lora_A.default.weight

Sample frozen layers (base model):
  🔒 base_model.model.transformer.wte.weight
  🔒 base_model.model.transformer.wpe.weight
  🔒 base_model.model.transformer.h.0.ln_1.weight
  🔒 base_model.model.transformer.h.0.ln_1.bias
  🔒 base_model.model.transformer.h.0.attn.c_attn.base_layer.weight

✅ Base model parameters are FROZEN
✅ Only LoRA parameters will be updated during training

STARTING FINE-TUNING
Training quantized LoRA parameters while keeping base model frozen...

/anaconda/envs/azureml_py38/lib/python3.10/site-packages/torch/utils/data/dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss,Accuracy
1,0.834700,0.641234,0.624000
2,0.366300,0.359994,0.860000


/anaconda/envs/azureml_py38/lib/python3.10/site-packages/torch/utils/data/dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/anaconda/envs/azureml_py38/lib/python3.10/site-packages/torch/utils/data/dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/anaconda/envs/azureml_py38/lib/python3.10/site-packages/torch/utils/data/dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)



TRAINING COMPLETE


/anaconda/envs/azureml_py38/lib/python3.10/site-packages/torch/utils/data/dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)



Test Results:
  eval_loss: 0.3600
  eval_accuracy: 0.8600
  eval_runtime: 125.9343
  eval_samples_per_second: 3.9700
  eval_steps_per_second: 0.5000
  epoch: 2.9600

SAMPLE PREDICTIONS

Text                                                         Prediction      Confidence
-------------------------------------------------------------------------------------
This movie was absolutely fantastic! I loved every minute... Positive        92.2%
Terrible film, complete waste of time and money.             Negative        55.7%
It was okay, nothing special.                                Positive        74.5%

SAVING MODEL
✓ Model saved to './gpt2_qlora_4steps_adapters'
  (Only LoRA adapters are saved - very small file!)
